# Kiafura — ARC-AGI-2 demonstration entry (ARC Prize 2026 Paper Track linkage)

Five frozen operational memories + fail-closed emission gates + the two-output contract,
emitting predictions-or-named-abstention. **Target-blind:** only `train` pairs and `test[].input`
are read. **DEMONSTRATION OUTPUT — the competition score is not an objective.**

Code and memories: CC0 1.0 Universal (public domain). Pipeline is the frozen inference/gate chain with
light driver glue only (`kaggle-demo/provenance.json`: `no_algorithm_modification: true`).

Submission format: `submission.json`, `attempt_1`/`attempt_2` per test input, in input order;
a slot with no real prediction carries the venue's `[[0, 0], [0, 0]]` example fill.

In [ ]:
# Cell 1 — mounts and file discovery (layout-agnostic: Kaggle may mount flat or nested)
from pathlib import Path

INPUTS = Path('/kaggle/input')
print('mounts:', sorted(p.name for p in INPUTS.iterdir()))

# competition challenges: any *test*challenges*.json anywhere under the mounts
chal_files = sorted(INPUTS.rglob('*test*challenges*.json'))
assert chal_files, f'no challenges file found under {INPUTS}'
CHAL = chal_files[0]
COMP = CHAL.parent  # the packer globs this dir for challenges + sample submission
print('challenges:', CHAL, '| competition dir:', COMP)

WORK = Path('/kaggle/working')
# package: the dir that CONTAINS kaggle-demo/ (zip may be auto-extracted, nested or flat)
driver_files = [p for p in INPUTS.rglob('driver.py') if p.parent.name == 'kaggle-demo']
assert driver_files, f'kaggle-demo/driver.py not found under {INPUTS}'
PKG_ROOT = driver_files[0].parent.parent
assert (PKG_ROOT / 'kaggle-submit' / 'make_submission.py').exists(), f'packer not found under {PKG_ROOT}'
print('package root:', PKG_ROOT)


In [ ]:
# Cell 2 — run the packer as a subprocess (signal.SIGALRM in the frozen driver is
# fragile inside an IPython kernel) and publish /kaggle/working/submission.json
import json, shutil, subprocess, sys

OUT = WORK / 'outputs'
packer = PKG_ROOT / 'kaggle-submit' / 'make_submission.py'
cmd = [sys.executable, str(packer),
       '--package-dir', str(PKG_ROOT / 'kaggle-demo'),
       '--data-dir', str(COMP),
       '--out', str(OUT),
       '--time-budget-s', '36000']   # wall-clock guard: 10h inside the 12h CPU limit
print('running:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-4000:])
if res.stderr:
    print('STDERR tail:', res.stderr[-2000:])
assert res.returncode == 0, 'packer failed'
shutil.copy(OUT / 'submission.json', WORK / 'submission.json')
print('submission.json bytes:', (WORK / 'submission.json').stat().st_size)

In [ ]:
# Cell 3 — validation against the venue contract
sub = json.loads((WORK / 'submission.json').read_text())
chal = json.loads(CHAL.read_text())
assert set(sub) == set(chal), 'task-id mismatch between submission and challenges'
for tid, slots in sub.items():
    assert len(slots) == len(chal[tid]['test']), f'{tid}: slot-count mismatch'
    for s in slots:
        assert set(s) == {'attempt_1', 'attempt_2'}, f'{tid}: missing attempt keys'
print('VALID:', len(sub), 'tasks;', sum(len(v) for v in sub.values()), 'test-input slots')
